# Phase 4: final report

Reads the completed outputs of Phases 0 through 3 and writes the report,
tables, figures, and artifact hashes to `phase4/outputs/`. Estimates and
intervals come before significance labels; confirmatory and exploratory
results stay separated; a stopped Phase 3 gate is reported without a C4 value.
The notebook needs no GPU. It runs on the pod at the end of the session, so
the whole experiment finishes in one place on one recorded machine.


In [1]:
%run definitions.ipynb
require_pythonhashseed()
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PH0 = PROJECT_ROOT / "phase0"
PH1 = PROJECT_ROOT / "phase1" / "outputs"
PH2 = PROJECT_ROOT / "phase2" / "outputs"
PH3 = PROJECT_ROOT / "phase3" / "outputs"
PH4 = PROJECT_ROOT / "phase4" / "outputs"
PH4.mkdir(parents=True, exist_ok=True)
ENV = environment_record()

# Protocol section 10: a change counts as a protocol deviation only after
# registered training begins. Nothing has trained, so this list starts empty.
# Append dated entries only for changes made after the first registered run.
EXPERIMENT_CHANGES = []

REQUIRED = {
    "phase0": PH0 / "artifacts" / "provenance.json",
    "phase1_tests": PH1 / "model_tests.json",
    "phase1_kernel": PH1 / "kernel_study.csv",
    "phase2_analysis": PH2 / "analysis_c1_c3.json",
    "phase3_gate": PH3 / "gate_result.json",
    "phase3_c4": PH3 / "analysis_c4.json",
}
missing = [k for k, p in REQUIRED.items() if not p.exists()]
P4_READY = not missing
if missing:
    print("PHASE 4 PENDING: missing inputs:", missing)
else:
    print("all phase inputs present")


required packages already match the pins.
torch 2.8.0+cu128 present (training phases need >= 2.8.0).
torchvision 0.23.0+cu128 present (training phases need >= 0.23.0).


python 3.12.3 | numpy 2.2.6 | pandas 2.3.3 | pillow 12.0.0 | opencv 4.12.0 | torch available: True


all phase inputs present


In [2]:
if P4_READY:
    provenance = json.load(open(REQUIRED["phase0"]))
    prevalence = pd.read_csv(PH0 / "reports" / "split_prevalence.csv")
    run_metrics = pd.read_csv(PH2 / "run_metrics.csv")
    p2 = json.load(open(REQUIRED["phase2_analysis"]))
    gate = json.load(open(REQUIRED["phase3_gate"]))
    c4 = json.load(open(REQUIRED["phase3_c4"]))

    confirmatory = pd.DataFrame(p2["confirmatory_C1_C2_C3"])
    if c4["estimated"]:
        confirmatory = pd.concat([confirmatory, pd.DataFrame([c4["C4"]])], ignore_index=True)
    else:
        confirmatory = pd.concat([confirmatory, pd.DataFrame([{
            "contrast": "C4", "estimate": None, "ci_low": None, "ci_high": None,
            "p_raw": None, "p_adjusted": None, "replicates": None}])], ignore_index=True)
    confirmatory.to_csv(PH4 / "confirmatory_table.csv", index=False)

    fold_auc = pd.DataFrame(p2["fold_macro_auc"])
    fold_auc.index = [f"fold {i}" for i in range(1, N_FOLDS + 1)]
    fold_auc.to_csv(PH4 / "fold_macro_auc.csv")

    cell_table = run_metrics.groupby("cell").agg(
        test_macro_auc_mean=("test_macro_auc", "mean"),
        test_macro_auc_sd=("test_macro_auc", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        hamming_mean=("hamming_loss", "mean"),
        exact_match_mean=("exact_match", "mean"),
        wall_seconds_mean=("wall_seconds", "mean"),
        trainable_parameters=("trainable_parameters", "max"),
    ).round(4)
    cell_table.to_csv(PH4 / "cell_summary.csv")

    exploratory = pd.DataFrame(p2["exploratory_capacity"])
    exploratory.to_csv(PH4 / "exploratory_capacity.csv", index=False)
    kernel = pd.read_csv(REQUIRED["phase1_kernel"])
    kernel.to_csv(PH4 / "kernel_study.csv", index=False)
    print(confirmatory.to_string(index=False))
    print()
    print(cell_table.to_string())
else:
    print("PENDING")


  ci_high    ci_low contrast  estimate  p_adjusted    p_raw  replicates
 0.013579  0.001243       C1  0.007534    0.007999 0.002000       10000
-0.004442 -0.016435       C2 -0.010361    0.000400 0.000100       10000
 0.018294 -0.005061       C3  0.006575    0.652735 0.163184       10000
-0.001972 -0.010619       C4 -0.006277    0.003600 0.000900       10000

       test_macro_auc_mean  test_macro_auc_sd  macro_f1_mean  hamming_mean  exact_match_mean  wall_seconds_mean  trainable_parameters
cell                                                                                                                                 
A                   0.9089             0.0084         0.6815        0.1218            0.6226             509.98                741162
B                   0.8952             0.0088         0.6499        0.1355            0.5836             530.06                740181
C                   0.9131             0.0107         0.6710        0.1173            0.6347           

In [3]:
if P4_READY:
    if c4["estimated"]:
        eff = {int(k): v for k, v in c4["effects_by_n"].items()}
        ns = sorted(eff)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(np.log10(ns), [eff[n] for n in ns], marker="o")
        ax.axhline(0, lw=0.8, color="gray")
        ax.set_xlabel("log10(training images)")
        ax.set_ylabel("DKA effect on macro AUC")
        est = c4["C4"]
        ax.set_title(f"C4 slope {est['estimate']:.4f} "
                     f"[{est['ci_low']:.4f}, {est['ci_high']:.4f}]")
        fig.tight_layout()
        fig.savefig(PH4 / "fig_c4_effect_vs_n.png", dpi=200)
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4))
    cells_order = ["A", "B", "C", "D", "E", "F"]
    for i, cll in enumerate(cells_order):
        vals = [p2["fold_macro_auc"][cll][str(f)] if isinstance(list(p2["fold_macro_auc"][cll])[0], str)
                else p2["fold_macro_auc"][cll][f] for f in range(1, N_FOLDS + 1)]
        ax.scatter([i] * N_FOLDS, vals, s=18)
        ax.scatter([i], [np.mean(vals)], marker="_", s=600, color="black")
    ax.set_xticks(range(len(cells_order)), cells_order)
    ax.set_xlabel("cell")
    ax.set_ylabel("test macro AUC per fold")
    fig.tight_layout()
    fig.savefig(PH4 / "fig_fold_auc_by_cell.png", dpi=200)
    plt.close(fig)
    print("figures written")
else:
    print("PENDING")


figures written


In [4]:
if P4_READY:
    manifest2 = manifest_frame(PH2)
    manifest3 = manifest_frame(PH3)
    failures = pd.concat([manifest2, manifest3])
    failures = failures[failures["status"] != "success"]
    failures.to_csv(PH4 / "failed_runs.csv", index=False)
    pd.DataFrame(EXPERIMENT_CHANGES).to_csv(PH4 / "experiment_changes.csv", index=False)

    lines = []
    lines.append("# Retinal adaptation study: final report\n")
    lines.append(f"Generated {ENV['timestamp_utc']} from the frozen protocol "
                 "`retinal_adaptation_study_v1.md`.\n")
    lines.append("## Cohort and data rules\n")
    lines.append(f"The cohort holds {EXPECTED_COHORT} images from the cleaned 2,208-row "
                 "MuReD table under the registered five-target rule. "
                 f"cohort.csv sha256: {provenance['cohort_sha256']}. "
                 "Splits, subsets, and audits are in phase0/.\n")
    lines.append("Data note: before Phase 0 ran, 221 RFMiD files disappeared from the "
                 "local images folder and were restored from the official images.zip "
                 "with CRC checks (phase0/source/restored_files.csv). No split, subset, "
                 "or model existed at that time.\n")
    lines.append("## Confirmatory results (C1-C4, Bonferroni across four tests)\n")
    lines.append("Estimates and 98.75% intervals come before significance labels. "
                 "The bootstrap is conditional on the trained models, the registered "
                 "seeds, and the fixed fold assignment; it covers held-out "
                 "analysis-group sampling only, and a non-significant result is not "
                 "equivalence.\n")
    lines.append("```text\n" + confirmatory.to_string(index=False) + "\n```\n")
    if not c4["estimated"]:
        lines.append("C4 is not estimated: the Phase 3 stability gate stopped the sweep "
                     f"(SD(50) = {gate['sd50']:.4f} against 0.030). C1-C3 keep the "
                     "four-test correction.\n")
    lines.append("## Cells at 1,044 images (fold-averaged)\n")
    lines.append("```text\n" + cell_table.to_string() + "\n```\n")
    lines.append("## Exploratory capacity contrasts (no family correction)\n")
    lines.append("```text\n" + exploratory.to_string(index=False) + "\n```\n")
    lines.append("## Kernel study (exploratory, inner validation only)\n")
    lines.append("```text\n" + kernel.to_string(index=False) + "\n```\n")
    lines.append("## Failures, reruns, and changes\n")
    lines.append(f"Failed or rerun rows: {len(failures)} (failed_runs.csv). "
                 f"Changes after registered training began: {len(EXPERIMENT_CHANGES)} "
                 "(experiment_changes.csv).\n")
    lines.append("## Scope\n")
    lines.append("Inference covers this cohort and this fold assignment. Fold effects "
                 "describe variation; the outer training pools overlap, so they are "
                 "not five independent replications.\n")
    (PH4 / "report.md").write_text("\n".join(lines))
    print(f"report.md written, {len(lines)} blocks")
else:
    print("PENDING")


report.md written, 18 blocks


In [5]:
if P4_READY:
    artifacts = sorted(p for p in PH4.iterdir() if p.is_file() and p.name != "artifact_hashes.json")
    hashes = {p.name: sha256_file(p) for p in artifacts}
    write_json(PH4 / "artifact_hashes.json", hashes)
    print(pd.Series(hashes).to_string())

    sections = ["Cohort and data rules", "Confirmatory results", "Cells at 1,044 images",
                "Exploratory capacity", "Kernel study", "Failures, reruns, and changes",
                "Scope"]
    text = (PH4 / "report.md").read_text()
    missing_sections = [s for s in sections if s not in text]
    if missing_sections:
        raise PhaseStop(f"report sections missing: {missing_sections}")
    print("\nPHASE 4 COMPLETE")
else:
    print("PHASE 4 PENDING")


cell_summary.csv            4ff9104b9ad4c6de16289e35db286b943a0dd8a8bf3118...
confirmatory_table.csv      e7645b66ad9723fbcd44d106013abd8e9bf655ff25cabc...
experiment_changes.csv      01ba4719c80b6fe911b091a7c05124b64eeece964e09c0...
exploratory_capacity.csv    9e57aefe1868a3e7b20bf4f3f0e5cc7d9faf2c634cb790...
failed_runs.csv             3506cc253e9b2282d567c5592058afcaec7633d42934e5...
fig_c4_effect_vs_n.png      8db3c6ac07356971d08fcabcb9db03bf2c27c9f8bb1cb2...
fig_fold_auc_by_cell.png    2b317b58563c870d541505add5ccaf844ac97b16dc9296...
fold_macro_auc.csv          b874a049e4c9ac3ac84dadc6696a9dd3e4e952a195d3d8...
kernel_study.csv            bdb492b1b53a55566a6ec7dc042d53a7db7a5ad69ffcc9...
report.md                   500d7f576868a2a17331eeb38bbd6ab8eaf539e07eded5...

PHASE 4 COMPLETE
